# Week 10 Lab: A Decision Tree by Hand

You'll build the core of a decision tree yourself, using the tax-cheating data from the lecture: first the
Gini impurity of a split, then a search for the best split, then a small tree.

**Core**: Exercises 1–3 (about 40 minutes). **Extension**: Exercise 4 (optional).

Run this first: it creates the data used in every exercise.

In [ ]:
import pandas as pd
import numpy as np

data = {
    'Refund': ['Yes', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No'],
    'Marital Status': ['Single', 'Married', 'Single', 'Married', 'Divorced', 'Married', 'Divorced', 'Single', 'Married', 'Single'],
    'Taxable Income (K)': [125, 100, 70, 120, 95, 60, 220, 85, 75, 90],
    'Cheat': ['No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes']
}
df = pd.DataFrame(data)
df

#### **Exercise 1**

**Gini impurity.** Complete `gini_impurity`. A split produces groups of rows; for each group, compute
1 − Σ p² (p = the share of the group in each class), then average the groups' values, weighting each by its size.

_Hint_: `['apple', 'pear', 'apple'].count('apple')` returns 2.

In [ ]:
def gini_impurity(groups, classes):
    """Weighted Gini impurity of a split.

    groups  -- a list of groups; each group is a list of the class labels that landed in it,
               e.g. [[True, False, True], [False, False, True]]
    classes -- the possible class labels, e.g. [True, False] or ['Yes', 'No']

    For each group: impurity = 1 - (sum of p**2 over the classes), where p is the share of the
    group in that class. The result is the average of the groups' impurities, weighted by group size.
    """
    gini = 0.0
    # YOUR CODE HERE
    return gini

gini_impurity([[True, False, True], [False, False, True]], [True, False])  # should be 0.4444

#### **Exercise 2**

**The best split.** The helpers below split the data on a column and generate candidate split points
(every value of a categorical column; the quartiles of a numeric one). Complete the loop in
`find_best_split`, then run it.

In [ ]:
def split_data(df, column, split_value, target):
    """Split df on column: numeric columns go left if <= split_value, categorical ones if == split_value.
    Returns two lists holding the target values of the left and right groups."""
    if pd.api.types.is_numeric_dtype(df[column]):
        mask = df[column] <= split_value
    else:
        mask = df[column] == split_value
    return df[mask][target].tolist(), df[~mask][target].tolist()


def get_quartiles(series):
    """Candidate split points for a numeric column: its 25th, 50th, and 75th percentiles."""
    return np.percentile(series, [25, 50, 75])

In [ ]:
def find_best_split(df, target):
    """Try every candidate split of every column (except target) and return the best one."""
    class_values = list(df[target].unique())
    best_column, best_value, best_gini, best_groups = None, None, float('inf'), None

    for column in [c for c in df.columns if c != target]:
        if pd.api.types.is_numeric_dtype(df[column]):
            split_candidates = get_quartiles(df[column])
        else:
            split_candidates = df[column].unique()

        for value in split_candidates:
            # YOUR CODE HERE: split the data on (column, value), compute its gini impurity,
            # and if it is lower than best_gini, update best_column, best_value, best_gini, best_groups
            pass

    return {'column': best_column, 'value': best_value, 'gini': best_gini, 'groups': best_groups}

best_split = find_best_split(df, 'Cheat')
print('Best split: column', best_split['column'], 'with value', best_split['value'],
      'gini =', round(best_split['gini'], 3))

**Question:** Which split is best, and what is its Gini impurity? In the lecture we worked out the
`Refund` split by hand (0.343). Why did your code choose a different one?

_Answer here_

#### **Exercise 3**

**A whole tree.** `build_tree` (given) grows the tree by calling your `find_best_split` again on each
child. Run it. Then use the printed rules to predict `Cheat` for each of the 10 rows, by hand, and compute
the tree's accuracy on these rows.

In [ ]:
def build_tree(df, target, depth=0, max_depth=3, min_samples=2):
    """Grow a tree by repeatedly calling find_best_split, and print it as nested rules."""
    indent = '    ' * depth
    # All rows in one class: this is a leaf
    if df[target].nunique() == 1:
        print(indent + f"Predict: {df[target].iloc[0]}")
        return
    # Too deep or too few rows: leaf with the majority class
    if depth >= max_depth or len(df) < min_samples:
        print(indent + f"Predict: {df[target].mode().iloc[0]}")
        return

    split = find_best_split(df, target)
    column, value = split['column'], split['value']
    numeric = pd.api.types.is_numeric_dtype(df[column])
    left_mask = df[column] <= value if numeric else df[column] == value

    print(indent + (f"If {column} <= {value}:" if numeric else f"If {column} == {value}:"))
    build_tree(df[left_mask], target, depth + 1, max_depth, min_samples)
    print(indent + (f"If {column} > {value}:" if numeric else f"If {column} != {value}:"))
    build_tree(df[~left_mask], target, depth + 1, max_depth, min_samples)

build_tree(df, 'Cheat')

In [ ]:
# Your predictions for rows 0-9, in order, read off the printed tree
my_predictions = []

# YOUR CODE HERE: compute the accuracy of my_predictions against df['Cheat']

**Question:** Is this accuracy a good estimate of how well the tree would do on new taxpayers? Why or
why not?

_Answer here_

---

### Extension exercise (optional)

#### **Exercise 4** (extension)

**Compare with scikit-learn.** Fit a `DecisionTreeClassifier(max_depth=3)` to the same data, as in the
lecture's scikit-learn tutorial, and print its rules with `export_text`. Use `pd.get_dummies` to one-hot
encode `Refund` and `Marital Status` first. Does scikit-learn choose the same first split as your code?
If not, why might it differ? (Think about which candidate split points each one tries.)

In [ ]:
# YOUR CODE HERE

_Answer here_